## Enhanced Multi-Previous-View Support + Clean Anchor

**New Features**: 
1. `include_prev` now supports **integer values** for multiple previous views (augmented copies of t-1).
2. **Clean t0 anchor**: One unaugmented t0 view is preserved for proper probe training
3. **Anchor-based invariance**: SSL invariance uses clean t0 as anchor instead of mean

### View Structure (Improved):
With `include_prev=2` and `repeat_factor=10`:

```
[t-1_aug1, t-1_aug2, t0_clean, t0_aug1, t0_aug2, ..., t0_aug10]
   ↑          ↑         ↑          ↑
   │          │         │          └── Augmented views for SSL invariance
   │          │         └── Clean/unaugmented t0 for PROBE training (ground truth)
   │          └── Augmented copy of the same t-1 window
   └── Augmented copy of the same t-1 window
```

### Configuration Options:
| `include_prev` | Clean t0 Index | Total Views | View Structure |
|----------------|----------------|-------------|----------------|
| 0 | 0 | 11 | `[t0_clean, t0_aug1, ..., t0_aug10]` |
| 1 | 1 | 12 | `[t-1_aug1, t0_clean, t0_aug1, ..., t0_aug10]` |
| 2 | 2 | 13 | `[t-1_aug1, t-1_aug2, t0_clean, t0_aug1, ..., t0_aug10]` |
| 3 | 3 | 14 | `[t-1_aug1, t-1_aug2, t-1_aug3, t0_clean, t0_aug1, ..., t0_aug10]` |

### Key Improvements:
- **Previous views are AUGMENTED copies of t-1** -> Stronger invariance signal without implicit time ordering
- **Clean t0 for probe** -> No augmentation noise in forecasting supervision
- **Anchor-based SSL** -> More stable invariance learning

This enables richer temporal context for SSL learning.

## 🔬 SSL Notes (Pure Self-Supervised)

This run uses the core LeJEPA SSL losses only:
- **Invariance** (anchor-based)
- **SIGReg** regularization

The predictor head and covariance regularization have been removed as requested.

### Embedding Health Diagnostics
These metrics remain for monitoring representation collapse:
- `embedding_std`: Mean std across features (should be ~1.0)
- `feature_collapse_ratio`: Fraction of features with std < 0.1 (should be ~0)

### Loss Structure
```
total_loss = (1 - λ) * inv_loss + λ * sigreg_loss
```

### Key Hyperparameters
| Parameter | Default | Description |
|-----------|---------|-------------|
| `lamb` | 0.5 | Balance inv vs sigreg |
| `proj_dim` | 64 | Projection dimension |


# LeJEPA SSL Framework - New Architecture Test

This notebook demonstrates the refactored SSL framework following the implementation plan:

## Key Features:
1. **Encoder-Agnostic**: Easily swap LSTM/CNN/Transformer/GNN encoders
2. **Past-Only SSL**: No t+1 views in SSL training (only t-1 and t0)
3. **Modular Data Pipeline**: WindowDataset -> ViewDataset -> Batch
4. **LeJEPA SSL Core**: Multi-view invariance + SIGReg regularization
5. **Detached Probe**: Forecasting evaluation without encoder gradients
6. **Scalable Augmentations**: Config-driven transforms (Ray Tune ready)

## Architecture:
- **SSL Core**: `LeJEPA_SSL` (encoder + projector + SIGReg)
- **Data**: `PeMS08SSLDataModule` using `WindowDataset` + `ViewDataset`
- **Training**: `SSLPretrainModule` Lightning wrapper
- **Probe**: Optional `ForecastProbe` for evaluation

View structure: `[t-1_aug..., t0_clean, t0_aug1, t0_aug2, ..., t0_augN]`

In [21]:
import os
import sys
from omegaconf import DictConfig
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

# Ensure project root is in path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# NEW imports: use the refactored SSL architecture
from src.timeseries.data.pems import PeMS08, PeMS08SSLDataModule
from src.timeseries.ssl.lejepa import LeJEPA_SSL
from src.timeseries.train.lightning_ssl import SSLPretrainModule
from src.timeseries.tasks.forecast_probe import ForecastProbe
from src.timeseries.encoders.rnn import GRUEncoder, LSTMEncoder
from src.timeseries.encoders.cnn import CNNEncoder
from src.timeseries.encoders.transformer import TransformerEncoder
from src.timeseries.encoders.upernet import UPerNetEncoder
from torchvision.ops import MLP
import matplotlib.pyplot as plt
from src.timeseries.visualizations.callbacks import VisualizationCallback

In [22]:
cfg = DictConfig({
    'window_size': 96,
    'target_window_size': 12,
    'prev_shift': 12,  # Shift between consecutive previous windows
    'stride': 1,
    'batch_size': 128,
    'num_workers': 16,
    'lr': 3e-4,
    'lamb': 0.5,  # Balance: 0.5 = equal inv + sigreg
    'epochs': 1000,
    'accelerator': 'auto',
    
    # SSL view configuration
    'repeat_factor': 10,  # Number of t0 augmented views
    'include_prev': 3,  # Number of augmented t-1 views
    
    # Encoder choice: 'upernet', 'transformer', 'cnn', 'gru', 'lstm'
    'encoder_backbone': 'transformer',
    
    # UPerNet encoder config
    'upernet_output_dim': 256,
    'upernet_stem_channels': 32,
    'upernet_pool_scales': (1, 2, 4),
    'upernet_dropout': 0.1,
    'upernet_channel_mixer': 'attn',
    'upernet_norm_type': 'batch',        # 'batch' (fastest), 'group', or 'layer' (original)
    'upernet_depthwise_sep': True,       # Depthwise-separable convs in backbone & FPN
    'upernet_num_stages': 4,             # 2–4 hierarchical stages
    'upernet_channel_mult': 2.0,         # Width multiplier per stage
    'upernet_upsample_mode': 'nearest',  # 'nearest' (fast) or 'linear' (smooth)
    
    # Transformer encoder config
    'transformer_output_dim': 512,
    'transformer_n_heads': 4,
    'transformer_n_layers': 1,
    'transformer_dim_feedforward': 512,
    'transformer_dropout': 0.1,
    'transformer_channel_mixer': 'attn',
    
    # CNN encoder config
    'cnn_output_dim': 512,
    'cnn_stem_channels': 128,
    'cnn_kernel_size': 5,
    'cnn_dilations': (1, 2, 4, 8),
    'cnn_dropout': 0.1,
    'cnn_channel_mixer': 'attn',
    
    # RNN encoder config
    'rnn_output_dim': 512,
    'rnn_hidden_channels': 256,  # Was 64 default — match capacity of other encoders
    'rnn_num_layers': 3,
    'rnn_dropout': 0.1,
    'rnn_channel_mixer': 'attn',  # Was 'none' — match transformer/CNN
    
    # Probe config
    'probe_hidden_dim': 512,
    'probe_num_hidden_layers': 4,
    'probe_dropout': 0.2,
    'probe_lr': 1e-3,
    'probe_start_epoch': 0,
    
    # Data split
    'split_mode': 'temporal',  # 'temporal', 'random_windows', or 'ts_cv'
    'train_split': 0.8,
    
    # Augmentation settings (tune for optimal invariance)
    'scale_range': (0.8, 1.2),  # Amplitude scaling range
    'crop_ratio_range': (0.8, 1.2),  # Keep t0 aligned (no temporal shift)
    'jitter_std': 0.1,  # Gaussian noise std
    'p_noise': 0.8,  # Probability of adding noise
    'p_freq_mask': 0.5,  # Probability of frequency masking
    'max_freq_ratio': 0.15,  # Max ratio of frequencies to mask
    'p_temporal_mask': 0.9,  # Probability of temporal block masking
    'p_magnitude_warp': 0.4,  # Smooth amplitude warp probability
    'p_temporal_crop': 0.0,  # Disable temporal crop to avoid view shift

    'p_transform': 0.99,  # Overall probability of applying transforms

})
print(
    f"View structure: {cfg.include_prev} prev + 1 clean + {cfg.repeat_factor} aug = "
    f"{cfg.include_prev + 1 + cfg.repeat_factor} views"
 )




View structure: 3 prev + 1 clean + 10 aug = 14 views


In [23]:
pems=PeMS08()
pems.target

,0,1,2,3,4,5,6,7,8,9,...,160,161,162,163,164,165,166,167,168,169
2016-07-01 00:00:00,133.0,210.0,124.0,145.0,206.0,58.0,248.0,117.0,44.0,256.0,...,168.0,168.0,13.0,61.0,102.0,133.0,38.0,74.0,94.0,6.0
2016-07-01 00:05:00,114.0,185.0,119.0,184.0,194.0,35.0,195.0,126.0,26.0,210.0,...,193.0,197.0,13.0,66.0,96.0,125.0,40.0,73.0,84.0,4.0
2016-07-01 00:10:00,140.0,171.0,107.0,146.0,162.0,106.0,205.0,120.0,55.0,224.0,...,180.0,168.0,9.0,52.0,99.0,130.0,40.0,70.0,82.0,4.0
2016-07-01 00:15:00,106.0,174.0,98.0,148.0,193.0,65.0,213.0,115.0,52.0,208.0,...,169.0,171.0,9.0,64.0,97.0,111.0,38.0,66.0,103.0,2.0
2016-07-01 00:20:00,117.0,176.0,114.0,130.0,163.0,68.0,206.0,115.0,39.0,188.0,...,186.0,166.0,7.0,88.0,87.0,127.0,35.0,60.0,86.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-08-31 23:35:00,131.0,191.0,149.0,140.0,213.0,59.0,232.0,111.0,44.0,192.0,...,159.0,177.0,105.0,65.0,89.0,76.0,8.0,100.0,100.0,1.0
2016-08-31 23:40:00,134.0,183.0,137.0,151.0,216.0,74.0,206.0,105.0,50.0,187.0,...,203.0,179.0,103.0,59.0,90.0,84.0,4.0,89.0,97.0,2.0
2016-08-31 23:45:00,120.0,176.0,119.0,142.0,194.0,68.0,237.0,105.0,40.0,180.0,...,174.0,153.0,87.0,61.0,90.0,91.0,15.0,47.0,91.0,3.0
2016-08-31 23:50:00,102.0,165.0,133.0,133.0,174.0,57.0,179.0,116.0,51.0,154.0,...,168.0,154.0,97.0,32.0,78.0,80.0,7.0,97.0,85.0,2.0


In [24]:
import plotly.express as px
n=100
px.line(pems.target[pems.target.columns[n:n+10]])

In [25]:
# Reload modified modules to pick up changes
import importlib
import src.timeseries.data.view_builders as vb
import src.timeseries.ssl.lejepa as ssl_lejepa
import src.timeseries.train.lightning_ssl as lightning_ssl
import src.timeseries.data.pems as pems_mod
importlib.reload(vb)
importlib.reload(ssl_lejepa)
importlib.reload(lightning_ssl)
importlib.reload(pems_mod)

# Re-import with reloaded modules
from src.timeseries.data.pems import PeMS08, PeMS08SSLDataModule
from src.timeseries.ssl.lejepa import LeJEPA_SSL
from src.timeseries.train.lightning_ssl import SSLPretrainModule

# Use the new PeMS08SSLDataModule which uses WindowDataset + ViewDataset
datamodule = PeMS08SSLDataModule(cfg)
datamodule.prepare_data()
datamodule.setup()

In [26]:
train_dataloader = datamodule.train_dataloader()
for batch in train_dataloader:
    # batch is now a Batch dataclass with views, view_times, targets, future_times
    print(f"Views shape: {batch.views.shape}")  # [B, V, C, T]
    print(f"View times shape: {batch.view_times.shape if batch.view_times is not None else None}")
    break

Views shape: torch.Size([128, 14, 170, 96])
View times shape: torch.Size([128, 14, 96, 6])


In [27]:
import importlib
import src.timeseries.visualizations.datamodule as dm
importlib.reload(dm)
from src.timeseries.visualizations.datamodule import visualize_pems_tuple

batch = next(iter(datamodule.train_dataloader()))
visualize_pems_tuple(
    batch,
    batch_index=0,
    sensor_index=0,
    window_size=cfg.window_size,
    target_window_size=cfg.target_window_size,
    temporal_shift=cfg.prev_shift,
    show_prev=True,
    show_next=False,
    show_target=True,
    max_t0_views=10,
    num_prev_views=cfg.include_prev,
    include_clean_t0=True,
 )

--- Visualization (train data) ---
Window lengths: L_in=96, Target=12. | Temporal shift: 12
Views: V=14 => prev=3, clean_t0=1, t0_aug=10


In [28]:
# Create encoder (choose backbone)
backbone = str(cfg.encoder_backbone).lower()
if backbone == 'upernet':
    encoder = UPerNetEncoder(
        input_channels=170,
        output_dim=cfg.upernet_output_dim,
        stem_channels=cfg.upernet_stem_channels,
        pool_scales=cfg.upernet_pool_scales,
        dropout=cfg.upernet_dropout,
        channel_mixer=cfg.upernet_channel_mixer,
        norm_type=cfg.upernet_norm_type,
        depthwise_sep=cfg.upernet_depthwise_sep,
        num_stages=cfg.upernet_num_stages,
        channel_mult=cfg.upernet_channel_mult,
        upsample_mode=cfg.upernet_upsample_mode,
    )
elif backbone == 'transformer':
    encoder = TransformerEncoder(
        input_channels=170,
        output_dim=cfg.transformer_output_dim,
        n_heads=cfg.transformer_n_heads,
        n_layers=cfg.transformer_n_layers,
        dim_feedforward=cfg.transformer_dim_feedforward,
        dropout=cfg.transformer_dropout,
        channel_mixer=cfg.transformer_channel_mixer,
    )
elif backbone == 'cnn':
    encoder = CNNEncoder(
        input_channels=170,
        output_dim=cfg.cnn_output_dim,
        stem_channels=cfg.cnn_stem_channels,
        kernel_size=cfg.cnn_kernel_size,
        dilations=cfg.cnn_dilations,
        dropout=cfg.cnn_dropout,
        channel_mixer=cfg.cnn_channel_mixer,
    )
elif backbone == 'gru':
    encoder = GRUEncoder(
        input_channels=170,
        output_dim=cfg.rnn_output_dim,
        hidden_channels=cfg.rnn_hidden_channels,
        num_layers=cfg.rnn_num_layers,
        dropout=cfg.rnn_dropout,
        channel_mixer=cfg.rnn_channel_mixer,
    )
elif backbone == 'lstm':
    encoder = LSTMEncoder(
        input_channels=170,
        output_dim=cfg.rnn_output_dim,
        hidden_channels=cfg.rnn_hidden_channels,
        num_layers=cfg.rnn_num_layers,
        dropout=cfg.rnn_dropout,
        channel_mixer=cfg.rnn_channel_mixer,
    )
else:
    raise ValueError(f"Unknown encoder_backbone: {cfg.encoder_backbone}")

# Create projector for SSL
proj_dim = 64  # Increased from 32 for better representation capacity
projector = MLP(in_channels=encoder.output_dim, hidden_channels=[512, proj_dim])

# Create SSL core module (no predictor, no covariance regularization)
ssl_core = LeJEPA_SSL(
    encoder=encoder,
    projector=projector,
    proj_dim=proj_dim,
    lamb=cfg.lamb,
    sigreg_slices=1024,
    sigreg_knots=17,
    num_prev_views=cfg.include_prev,
    include_clean_t0=True,
    use_anchor_invariance=False,
 )

# Optional: Create forecasting probe for evaluation (detached from encoder)
probe = ForecastProbe(
    input_dim=encoder.output_dim,
    horizon=cfg.target_window_size,
    output_channels=170,
    use_covariates=False,
    hidden_dim=cfg.probe_hidden_dim,
    num_hidden_layers=cfg.probe_num_hidden_layers,
    dropout=cfg.probe_dropout,
    level_fusion='attn',
 )

# Wrap in Lightning module
model = SSLPretrainModule(
    ssl_core=ssl_core,
    lr=cfg.lr,
    weight_decay=5e-2,
    probe=probe,
    probe_loss_weight=0.1,  # Probe loss for monitoring only
    probe_weight_decay=1e-4,
    probe_lr=cfg.probe_lr,
    probe_start_epoch=cfg.probe_start_epoch,
 )

print(f"SSL architecture initialized with {cfg.include_prev} previous views!")
print(f"Clean t0 at index: {ssl_core.clean_t0_index}")
print(f"Total views per batch: {cfg.include_prev + 1 + cfg.repeat_factor}")
print(f"Encoder has forward_multilevel: {hasattr(encoder, 'forward_multilevel')}")

SSL architecture initialized with 3 previous views!
Clean t0 at index: 3
Total views per batch: 14
Encoder has forward_multilevel: True


In [29]:
logger = TensorBoardLogger("tb_logs", name="lejepa_ssl_new")

callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints",
        filename="lejepa-ssl-{epoch:02d}-{val/ssl_loss:.4f}",
        monitor="val/ssl_loss",
        mode="min",
        save_top_k=3
    ),
    LearningRateMonitor(logging_interval="step"),
    VisualizationCallback(
        num_samples_plot=1,
        sensor_indices=[0, 7, 42],
        fixed_samples=True,      # keep the same sample rows each epoch
        sample_indices=[25],
        fixed_window_stage="both",
        )
]

trainer = L.Trainer(
    max_epochs=cfg.epochs,
    accelerator=cfg.accelerator,
    devices=1,
    logger=logger,
    callbacks=callbacks,
    log_every_n_steps=10,
    gradient_clip_val=0.30,
    gradient_clip_algorithm="norm",
    limit_train_batches=0.1,  # Set to <1.0 for quick testing, 1.0 for full training
    limit_val_batches=0.1,  # Set to <1.0 for quick testing, 1.0 for full validation
    
)

print("Trainer ready. Starting SSL pretraining...")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Trainer ready. Starting SSL pretraining...


In [30]:
trainer.fit(model, datamodule=datamodule)

/home/crispy/lejepa/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: UserWarning:

Checkpoint directory /home/crispy/lejepa/src/timeseries/checkpoints exists and is not empty.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type          | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | ssl_core | LeJEPA_SSL    | 2.2 M  | train | 0    
1 | probe    | ForecastProbe | 2.2 M  | train | 0    
-----------------------------------------------------------
4.4 M     Trainable params
0         Non-trainable params
4.4 M     Total params
17.653    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

RuntimeError: DataLoader worker (pid 1066536) is killed by signal: Killed. 